# Data Exploration: SEC EDGAR Filings

This notebook explores the raw SEC filings downloaded from EDGAR:
- Document structure and section identification
- Text statistics (length, vocabulary)
- Metadata quality checks
- Sample content from key sections

In [ ]:
import sys
sys.path.insert(0, '..')
import re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from modules.ingestion.parsers import SECFilingParser, SEC_10K_SECTIONS
from modules.ingestion.metadata import extract_from_edgar_path

plt.style.use('seaborn-v0_8-whitegrid')
DATA_DIR = Path('../data/raw/sec-edgar-filings')

In [ ]:
class DocumentMetadata:
    """Structured metadata for a financial document."""

    ticker: str
    filing_type: str
    filing_date: str | None = None
    source_url: str | None = None
    source_path: str | None = None
    company_name: str | None = None

    def to_dict(self) -> dict:
        """Convert to dict for storage, excluding None values."""
        return {k: v for k, v in self.__dict__.items() if v is not None}


In [ ]:
def extract_from_edgar_path(file_path: Path) -> DocumentMetadata:
    """This method extracts metadata from SEC EDGAR download directory structure.

    EDGAR downloader saves files in:
    data/raw/sec-edgar-filings/{TICKER}/{FILING_TYPE}/{ACCESSION}/...

    Args:
        file_path: Path to a downloaded filing file.

    Returns:
        DocumentMetadata with ticker and filing type from the path.
    """
    parts = file_path.parts
    metadata = DocumentMetadata(ticker="UNKNOWN", filing_type="UNKNOWN")

    # Look for the pattern: sec-edgar-filings / TICKER / FILING_TYPE
    for i, part in enumerate(parts):
        if part == "sec-edgar-filings" and i + 2 < len(parts):
            metadata.ticker = parts[i + 1].upper()
            metadata.filing_type = parts[i + 2]
            break

    metadata.source_path = str(file_path)

    # Try to extract filing date from the document filename or accession number
    date_match = re.search(r"(\d{4}-\d{2}-\d{2})", str(file_path))
    if date_match:
        metadata.filing_date = date_match.group(1)

    return metadata

In [ ]:
SEC_10K_SECTIONS = {
    "Item 1": "Business",
    "Item 1A": "Risk Factors",
    "Item 1B": "Unresolved Staff Comments",
    "Item 2": "Properties",
    "Item 3": "Legal Proceedings",
    "Item 4": "Mine Safety Disclosures",
    "Item 5": "Market for Registrant's Common Equity",
    "Item 6": "Reserved",
    "Item 7": "Management's Discussion and Analysis",
    "Item 7A": "Quantitative and Qualitative Disclosures About Market Risk",
    "Item 8": "Financial Statements and Supplementary Data",
    "Item 9": "Changes in and Disagreements With Accountants",
    "Item 9A": "Controls and Procedures",
    "Item 9B": "Other Information",
    "Item 10": "Directors, Executive Officers and Corporate Governance",
    "Item 11": "Executive Compensation",
    "Item 12": "Security Ownership",
    "Item 13": "Certain Relationships and Related Transactions",
    "Item 14": "Principal Accountant Fees and Services",
    "Item 15": "Exhibits and Financial Statement Schedules",
}


## List all available filings and their metadata.

In [ ]:
# Find all downloaded filing files
filing_files = list(DATA_DIR.rglob('*.htm')) + list(DATA_DIR.rglob('*.html')) + list(DATA_DIR.rglob('*.pdf'))  + list(DATA_DIR.rglob('*.txt'))
print(f'Found {len(filing_files)} filing files')

# Extracting metadata from each file path
records = []
for f in filing_files:
    meta = extract_from_edgar_path(f)
    records.append({
        'ticker': meta.ticker,
        'filing_type': meta.filing_type,
        'filing_date': meta.filing_date,
        'path': str(f),
    })

df_filings = pd.DataFrame(records)
df_filings.head(10)

In [ ]:
# Filing distribution by ticker and type
if not df_filings.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df_filings['ticker'].value_counts().plot.bar(ax=axes[0], title='Filings by Ticker')
    df_filings['filing_type'].value_counts().plot.bar(ax=axes[1], title='Filings by Type')
    plt.tight_layout()
    plt.show()
else:
    print("No filings found.")

## Parse a Sample Filing
section extraction from a 10-K filing

In [ ]:
class ParsedDocument:
    """A parsed financial document with extracted text and metadata."""

    text: str
    source_path: str
    doc_type: str  # "10-K", "10-Q", "PDF"
    sections: dict[str, str] = field(default_factory=dict)


In [ ]:
def parse(file_path: Path) -> ParsedDocument:
        """Parse an SEC filing HTML file and extract sections.

        Args:
            file_path: Path to the filing HTML file.

        Returns:
            ParsedDocument with full text and per-section text.
        """
        logger.info("Parsing SEC filing", path=str(file_path))
        raw_html = file_path.read_text(encoding="utf-8", errors="replace")
        soup = BeautifulSoup(raw_html, "lxml")

        # Remove script/style tags
        for tag in soup(["script", "style"]):
            tag.decompose()

        full_text = soup.get_text(separator="\n", strip=True)
        sections = self._extract_sections(full_text)

        return ParsedDocument(
            text=full_text,
            source_path=str(file_path),
            doc_type="10-K",
            sections=sections,
        )

In [ ]:
# Parse the first available filing
if filing_files:
    sample_doc = parse(filing_files[0])
    print(f'Parsed: {filing_files[0].name}')
    print(f'Total text length: {len(sample_doc.text):,} chars')
    print(f'Sections found: {len(sample_doc.sections)}')
    print()
    for section_name, section_text in sample_doc.sections.items():
        print(f'  {section_name}: {len(section_text):,} chars')
else:
    print('No filing files available to parse.')

In [ ]:
# Visualize section lengths
if filing_files and sample_doc.sections:
    section_df = pd.DataFrame([
        {'section': name, 'chars': len(text)}
        for name, text in sample_doc.sections.items()
    ])
    section_df = section_df.sort_values('chars', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    section_df.plot.barh(x='section', y='chars', ax=ax, title='Section Lengths (chars)')
    ax.set_xlabel('Characters')
    plt.tight_layout()
    plt.show()

## Sample Content Preview


In [ ]:
# Preview key sections
key_sections = ['Item 1A', 'Item 7']

if filing_files and sample_doc.sections:
    for section in key_sections:
        if section in sample_doc.sections:
            text = sample_doc.sections[section]
            print(f'\n{"="*60}')
            print(f'{section} - {SEC_10K_SECTIONS.get(section, "")} (first 500 chars)')
            print(f'{"="*60}')
            print(text[:500])
            print('...')

## Text Statistics
Analyze word counts, vocabulary richness, and common financial terms.

In [ ]:
from collections import Counter
import re

if filing_files:
    words = re.findall(r'\b\w+\b', sample_doc.text.lower())
    word_counts = Counter(words)
    
    print(f'Total words: {len(words):,}')
    print(f'Unique words: {len(word_counts):,}')
    print(f'Vocabulary richness: {len(word_counts)/len(words):.2%}')
    
    # Financial-specific terms
    financial_terms = [
        'revenue', 'income', 'loss', 'risk', 'debt', 'equity',
        'operating', 'assets', 'liabilities', 'cash', 'margin',
        'earnings', 'shares', 'dividend', 'impairment',
    ]
    print('\nFinancial term frequencies:')
    for term in financial_terms:
        count = word_counts.get(term, 0)
        if count > 0:
            print(f'  {term}: {count}')

## Summary
- Section extraction successfully identifies standard 10-K items
- Risk Factors and MD&A are typically the longest and most information-dense sections
- Financial terminology is well-represented across filings
